# Step 2 V2 - frozen positive calibration and one untouched final evaluation

This notebook loads only V2 Step 1 artifacts. All specifications, timing, lag order, and weekday treatment are frozen before the final-test outcomes are opened. Forecasts are fixed-parameter **rolling-origin one-step-ahead** forecasts: at each test date the origin advances and uses the realised information set through the preceding model day, while coefficients remain those estimated on training data.

Equation 3 targets the unshifted quote-date implied variance. It never reuses the Equation 2 RV-aligned IV column. OLS in log variance is Gaussian MLE only if the log errors are conditionally Gaussian with constant variance; under correct conditional mean and predetermined regressors it can remain consistent under weaker conditions, but is inefficient under heteroskedasticity, so HAC uncertainty is reported.

In [1]:
from pathlib import Path
import os,json,warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import GammaRegressor
from sklearn.mixture import GaussianMixture
from IPython.display import display

SEED=16017; rng=np.random.default_rng(SEED); ALPHA=.05
BASE=Path(os.environ.get("PIPELINE_BASE",Path.cwd())).resolve(); IN1=BASE/"step1_results_v2"; OUT=BASE/"step2_results_v2"; OUT.mkdir(parents=True,exist_ok=True)
panel=pd.read_csv(IN1/"step1_panel_v2.csv",parse_dates=["model_day"]); dec=json.loads((IN1/"frozen_decisions.json").read_text())
SPLIT=pd.Timestamp(dec["split_date"]); INNER=pd.Timestamp(dec["inner_validation_start"]); P=int(dec["primary_lag_order"]); TIMING=dec["chosen_eq2_timing"]; WEEKDAY=bool(dec["weekday_adjustment"])
assert dec["eq3_target"]=="unshifted iv_quote on quote date" and dec["eq3_regressor_rule"]=="all dataframe offsets <= -1"
assert panel.query("sample_role=='training'").model_day.max()<SPLIT<=panel.query("sample_role=='final_test'").model_day.min()
print(json.dumps(dec,indent=2))

{
  "seed": 16017,
  "split_date": "2022-04-12",
  "inner_validation_start": "2018-11-27",
  "final_test_rule": "model_day >= split_date; untouched until Step 2 final evaluation",
  "iv_source_timestamp": "absent; date only",
  "timing_variants": [
    "advance",
    "same_label"
  ],
  "chosen_eq2_timing": "advance",
  "eq3_target": "unshifted iv_quote on quote date",
  "eq3_regressor_rule": "all dataframe offsets <= -1",
  "primary_lag_order": 5,
  "weekday_adjustment": true,
  "selection_metric": "timing fixed to conservative advance information set; lag and weekday chosen by mean inner-validation QLIKE across Eq2 and Eq3 positive log models",
  "multiplicity": "Holm across 24 direction x lag x timing tests",
  "retained_predictive_directions": [
    "iv -> r2",
    "iv -> rv"
  ],
  "eq2_blocks_retained": [
    "iv",
    "rv"
  ],
  "eq3_blocks_retained": [
    "iv"
  ],
  "conditional_nonlinear_role": "exploratory, not retention gate",
  "equation5_forward_variance": "not implemen

## Frozen information sets and training designs

No test target is referenced in this section. `advance` Equation 2 uses quote-date D-j for session-D RV; Equation 3 always uses D-j states for the date-D quote. Lag order 5 means lags 1 through 5 jointly.

In [2]:
STATES=["r2","rv","iv"]
def offsets(src,tgt,p=P):
    if src=="iv" and tgt in ("rv","r2") and TIMING=="same_label": return list(range(0,p))
    return list(range(1,p+1))
def series(src): return panel.iv_quote if src=="iv" else panel[src]
def block(src,tgt,p=P,log=True):
    a=np.column_stack([series(src).shift(k) if k else series(src) for k in offsets(src,tgt,p)])
    if log:
        pos=a[np.isfinite(a)&(a>0)]; eps=float(np.min(pos)/2) if len(pos) else 1e-12
        a=np.log(np.maximum(a,eps))
    return a
WD=pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy()
def contiguous(maxoff):
    br=panel.contig_break.to_numpy(bool); ok=np.ones(len(panel),bool)
    for k in range(maxoff): ok &= ~np.r_[np.ones(k,bool),br[:len(br)-k]] if k else ~br
    ok[:maxoff]=False; return ok
RESTRICTED={"rv":dec["eq2_blocks_retained"],"iv":dec["eq3_blocks_retained"]}
def matrix(target,blocks,p=P,log=True,weekday=WEEKDAY):
    X=np.column_stack([block(s,target,p,log) for s in blocks]) if blocks else np.empty((len(panel),0))
    if weekday: X=np.column_stack([X,WD])
    mo=max([max(offsets(s,target,p)) for s in blocks] or [p]); y=(panel.iv_quote if target=="iv" else panel[target]).to_numpy(float)
    ok=contiguous(mo)&np.isfinite(y)&(y>0)&np.isfinite(X).all(1)
    return y,X,ok
def har_matrix(target):
    y=(panel.iv_quote if target=="iv" else panel[target]).to_numpy(float); s=pd.Series(y)
    own=np.column_stack([np.log(np.maximum(s.shift(1),1e-12)),np.log(np.maximum(s.shift(1).rolling(5).mean(),1e-12)),np.log(np.maximum(s.shift(1).rolling(22).mean(),1e-12))])
    X=np.column_stack([own,WD]) if WEEKDAY else own; ok=contiguous(22)&np.isfinite(y)&(y>0)&np.isfinite(X).all(1); return y,X,ok

train=panel.model_day.lt(SPLIT).to_numpy(); pre_inner=panel.model_day.lt(INNER).to_numpy(); inner=panel.model_day.ge(INNER).to_numpy()&train
fits={}; coef=[]
for target in ("rv","iv"):
  specs={"restricted_log":RESTRICTED[target],"unrestricted_log":STATES,"ar_log":[target]}
  for name,blocks in specs.items():
    y,X,ok=matrix(target,blocks); tr=ok&train
    m=sm.OLS(np.log(y[tr]),sm.add_constant(X[tr],has_constant="add")).fit(cov_type="HAC",cov_kwds={"maxlags":max(5,P)})
    fits[(target,name)]={"model":m,"blocks":blocks,"kind":"log"}
    for j,(b,se,pv) in enumerate(zip(m.params,m.bse,m.pvalues)): coef.append({"target":target,"model":name,"term":f"b{j}","coef":b,"se_HAC":se,"p_HAC":pv,"n_train":int(tr.sum())})
  y,X,ok=har_matrix(target); tr=ok&train; m=sm.OLS(np.log(y[tr]),sm.add_constant(X[tr],has_constant="add")).fit(cov_type="HAC",cov_kwds={"maxlags":22}); fits[(target,"har_log")]={"model":m,"kind":"har"}
  for j,(b,se,pv) in enumerate(zip(m.params,m.bse,m.pvalues)): coef.append({"target":target,"model":"har_log","term":f"b{j}","coef":b,"se_HAC":se,"p_HAC":pv,"n_train":int(tr.sum())})
  # Positive Gamma log-link restricted variant; standardisation is learned on training only.
  y,X,ok=matrix(target,RESTRICTED[target],log=False); tr=ok&train; mu=X[tr].mean(0); sd=X[tr].std(0); sd[sd==0]=1
  gm=GammaRegressor(alpha=0,max_iter=5000,tol=1e-9).fit((X[tr]-mu)/sd,y[tr]); fits[(target,"gamma_restricted")]={"model":gm,"blocks":RESTRICTED[target],"kind":"gamma","mu":mu,"sd":sd}
pd.DataFrame(coef).to_csv(OUT/"step2_coefficients.csv",index=False); display(pd.DataFrame(coef))

,target,model,term,coef,se_HAC,p_HAC,n_train
0,rv,restricted_log,b0,-3.912175,0.229192,2.507413e-65,3738
1,rv,restricted_log,b1,0.636602,0.039848,1.884720e-57,3738
2,rv,restricted_log,b2,-0.122517,0.034986,4.619049e-04,3738
3,rv,restricted_log,b3,0.053907,0.034096,1.138727e-01,3738
4,rv,restricted_log,b4,-0.031707,0.035787,3.756237e-01,3738
...,...,...,...,...,...,...,...
96,iv,har_log,b3,0.213320,0.034335,5.198064e-10,3907
97,iv,har_log,b4,-0.289118,0.029512,1.165676e-22,3907
98,iv,har_log,b5,-0.253143,0.030591,1.282220e-16,3907
99,iv,har_log,b6,-0.156943,0.032004,9.399643e-07,3907


## Equation 1: drift, standardised innovations, density, tails, and variance calibration

Diagnostics and density-model choice use training/inner validation only. The optional jump comparison is a reduced-form two-Gaussian rare-component mixture for standardised innovations; it is not asserted to identify the PDF's Poisson jump mechanism.

In [3]:
tr1=train&panel.rv_eligible.to_numpy(bool)&panel.r.notna().to_numpy()&panel.rv.notna().to_numpy()&(panel.rv.to_numpy()>0)
r=panel.r.to_numpy(float)[tr1]; rv=panel.rv.to_numpy(float)[tr1]; dates=panel.model_day.to_numpy()[tr1]
mu=float(r.mean()); hac=sm.OLS(r,np.ones(len(r))).fit(cov_type="HAC",cov_kwds={"maxlags":10}); z=(r-mu)/np.sqrt(rv)
jb=stats.jarque_bera(z); nt=stats.normaltest(z)
cal=sm.OLS(r**2,sm.add_constant(rv)).fit(cov_type="HAC",cov_kwds={"maxlags":10})
eq1=pd.DataFrame([{"mu_hat":mu,"mu_HAC_se":float(hac.bse[0]),"mu_HAC_p":float(hac.pvalues[0]),"z_mean":z.mean(),"z_sd":z.std(ddof=1),
 "skew":stats.skew(z),"excess_kurtosis":stats.kurtosis(z),"JB_p":jb.pvalue,"normaltest_p":nt.pvalue,
 "tail_abs_gt2":np.mean(np.abs(z)>2),"tail_abs_gt3":np.mean(np.abs(z)>3),"variance_cal_intercept":cal.params[0],"variance_cal_slope":cal.params[1],"variance_cal_slope_p":cal.pvalues[1]}])
eq1.to_csv(OUT/"step2_equation1_training_diagnostics.csv",index=False); display(eq1)
pre=pd.to_datetime(dates)<INNER; va=~pre
g1=GaussianMixture(1,random_state=SEED).fit(z[pre,None]); g2=GaussianMixture(2,random_state=SEED,n_init=10).fit(z[pre,None])
score1=float(g1.score(z[va,None])); score2=float(g2.score(z[va,None])); chosen_components=2 if score2>score1 else 1
gfinal=GaussianMixture(chosen_components,random_state=SEED,n_init=10).fit(z[:,None])
density_choice={"inner_gaussian_logscore":score1,"inner_jump_mixture_logscore":score2,"chosen_components":chosen_components,
 "interpretation":"two components are reduced-form jump robustness, not Poisson-jump identification"}
(OUT/"step2_return_density_choice.json").write_text(json.dumps(density_choice,indent=2)); print(density_choice)

,mu_hat,mu_HAC_se,mu_HAC_p,z_mean,z_sd,skew,excess_kurtosis,JB_p,normaltest_p,tail_abs_gt2,tail_abs_gt3,variance_cal_intercept,variance_cal_slope,variance_cal_slope_p
0,0.00006,0.000081,0.462192,0.02947,0.955551,0.059362,-0.44349,1.225162e-09,3.072111e-15,0.025893,0.000214,0.000003,0.878846,1.016009e-12


{'inner_gaussian_logscore': -1.4104026761356152, 'inner_jump_mixture_logscore': -1.4099548832222752, 'chosen_components': 2, 'interpretation': 'two components are reduced-form jump robustness, not Poisson-jump identification'}


## Noise scales in Equations 2-3

Scales are estimated for log-variance innovations. Constant, EWMA(0.94/0.97), and realised-quarticity-aware alternatives are compared on inner-training Gaussian log score. This affects uncertainty calibration, not the already frozen conditional-mean specification.

In [4]:
scale_rows=[]; scale_choice={}
for target in ("rv","iv"):
    y,X,ok=matrix(target,RESTRICTED[target]); pre=ok&pre_inner; inn=ok&inner
    m0=sm.OLS(np.log(y[pre]),sm.add_constant(X[pre],has_constant="add")).fit()
    all_idx=np.flatnonzero(ok); e=np.full(len(panel),np.nan); e[ok]=np.log(y[ok])-m0.predict(sm.add_constant(X[ok],has_constant="add"))
    candidates={"constant":np.full(len(panel),np.nanstd(e[pre],ddof=1)**2)}
    for lam in (.94,.97):
        v=np.full(len(panel),np.nan); last=np.nanvar(e[pre],ddof=1)
        for i in range(len(panel)):
            if np.isfinite(e[i]): last=lam*last+(1-lam)*e[i]**2
            v[i]=last
        candidates[f"EWMA_{lam:.2f}"]=v
    rq=np.log(np.maximum(panel.rq.shift(1).to_numpy(float),1e-20)); qok=pre&np.isfinite(rq)&np.isfinite(e)
    qm=sm.OLS(np.log(e[qok]**2+1e-8),sm.add_constant(rq[qok])).fit(); qv=np.exp(qm.predict(sm.add_constant(np.nan_to_num(rq,nan=np.nanmedian(rq[qok])))))
    candidates["RQ_aware"]=qv
    scores={}
    for name,v in candidates.items():
        use=inn&np.isfinite(e)&np.isfinite(v)&(v>0); sc=float(np.mean(-.5*(np.log(2*np.pi*v[use])+e[use]**2/v[use]))); scores[name]=sc
        scale_rows.append({"target":target,"scale_model":name,"inner_logscore":sc,"training_noise_sd":float(np.sqrt(np.nanmedian(v[pre]))),"inner_n":int(use.sum())})
    scale_choice[target]=max(scores,key=scores.get)
pd.DataFrame(scale_rows).to_csv(OUT/"step2_noise_scales.csv",index=False); (OUT/"step2_noise_scale_choice.json").write_text(json.dumps(scale_choice,indent=2)); display(pd.DataFrame(scale_rows)); print(scale_choice)

,target,scale_model,inner_logscore,training_noise_sd,inner_n
0,rv,constant,-1.015906,0.640933,841
1,rv,EWMA_0.94,-0.945370,0.611541,841
2,rv,EWMA_0.97,-0.976343,0.622810,841
3,rv,RQ_aware,-1.789841,0.314058,841
4,iv,constant,-0.411278,0.389679,877
5,iv,EWMA_0.94,-0.316899,0.355819,877
6,iv,EWMA_0.97,-0.352386,0.361286,877
7,iv,RQ_aware,-1.237369,0.176202,877


{'rv': 'EWMA_0.94', 'iv': 'EWMA_0.94'}


## Final untouched test - opened once

The next cell is the sole final-test evaluation. Every forecast is positive by construction. All models share the common HAR-admissible test dates for paired loss and DM inference. Random-walk and fitted models use actual lagged values observable at each rolling origin; no future value enters a regressor.

In [5]:
def qlike(y,f): z=y/f; return z-np.log(z)-1
def dm(l1,l2):
    d=np.asarray(l1)-np.asarray(l2); T=len(d); dc=d-d.mean(); L=int(np.floor(4*(T/100)**(2/9))); lrv=np.mean(dc**2)
    for k in range(1,L+1): lrv+=2*(1-k/(L+1))*np.mean(dc[k:]*dc[:-k])
    if lrv<=0:return np.nan,np.nan
    st=d.mean()/np.sqrt(lrv/T); return float(st),float(2*stats.t.sf(abs(st),T-1))
metrics=[]; forecasts=[]; dmrows=[]
for target in ("rv","iv"):
    y=(panel.iv_quote if target=="iv" else panel[target]).to_numpy(float)
    _,_,harok=har_matrix(target); common=harok&panel.model_day.ge(SPLIT).to_numpy()&np.isfinite(y)&(y>0)
    for nm in ("restricted_log","unrestricted_log","ar_log"):
        common &= matrix(target,fits[(target,nm)]["blocks"])[2]
    common &= matrix(target,fits[(target,"gamma_restricted")]["blocks"],log=False)[2]
    common &= np.isfinite(series(target).shift(1).to_numpy(float))
    idx=np.flatnonzero(common)
    pred={}
    for name in ("restricted_log","unrestricted_log","ar_log"):
        info=fits[(target,name)]; _,X,ok=matrix(target,info["blocks"]); assert ok[idx].all(); pred[name]=np.exp(info["model"].predict(sm.add_constant(X[idx],has_constant="add")))
    _,Xh,_=har_matrix(target); pred["har_log"]=np.exp(fits[(target,"har_log")]["model"].predict(sm.add_constant(Xh[idx],has_constant="add")))
    info=fits[(target,"gamma_restricted")]; _,Xg,ok=matrix(target,info["blocks"],log=False); pred["gamma_restricted"]=info["model"].predict((Xg[idx]-info["mu"])/info["sd"])
    pred["random_walk"]=series(target).shift(1).to_numpy(float)[idx]
    actual=y[idx]
    for name,f in pred.items():
        assert np.isfinite(f).all() and (f>0).all(); e=actual-f
        metrics.append({"target":target,"model":name,"n_test":len(idx),"test_start":str(panel.model_day.iloc[idx[0]].date()),"test_end":str(panel.model_day.iloc[idx[-1]].date()),"RMSE":np.sqrt(np.mean(e**2)),"MAE":np.mean(np.abs(e)),"QLIKE":np.mean(qlike(actual,f))})
        forecasts.extend({"model_day":panel.model_day.iloc[i],"target":target,"model":name,"actual":a,"forecast":ff} for i,a,ff in zip(idx,actual,f))
    for rival in ("unrestricted_log","ar_log","har_log","random_walk","gamma_restricted"):
      for loss,fun in {"SE":lambda a,f:(a-f)**2,"AE":lambda a,f:np.abs(a-f),"QLIKE":qlike}.items():
        st,pv=dm(fun(actual,pred["restricted_log"]),fun(actual,pred[rival])); dmrows.append({"target":target,"restricted_vs":rival,"loss":loss,"DM_stat":st,"p_value":pv,"nested_comparison":rival in ("unrestricted_log","ar_log"),"interpretation":"descriptive for nested pair; standard DM can be conservative/nonstandard" if rival in ("unrestricted_log","ar_log") else "non-nested predictive comparison"})
met=pd.DataFrame(metrics); fc=pd.DataFrame(forecasts); dmt=pd.DataFrame(dmrows)
met.to_csv(OUT/"step2_final_test_metrics.csv",index=False); fc.to_csv(OUT/"step2_final_test_forecasts.csv",index=False); dmt.to_csv(OUT/"step2_dm_tests.csv",index=False)
assert fc.forecast.gt(0).all(); display(met); display(dmt)

# Equation 1 final density evaluation is also opened only here.
te=panel.model_day.ge(SPLIT).to_numpy()&panel.rv_eligible.to_numpy(bool)&panel.r.notna().to_numpy()&panel.rv.notna().to_numpy()&(panel.rv.to_numpy()>0)
zte=(panel.r.to_numpy(float)[te]-mu)/np.sqrt(panel.rv.to_numpy(float)[te]); gauss_ll=float(np.mean(stats.norm.logpdf(zte))); mix_ll=float(gfinal.score(zte[:,None]))
ret_eval={"n_test":int(te.sum()),"gaussian_logscore":gauss_ll,"selected_density_components":chosen_components,"selected_density_logscore":mix_ll,"tail_abs_gt3":float(np.mean(np.abs(zte)>3))}
(OUT/"step2_return_density_final_test.json").write_text(json.dumps(ret_eval,indent=2)); print(ret_eval)

,target,model,n_test,test_start,test_end,RMSE,MAE,QLIKE
0,rv,restricted_log,894,2022-04-12,2026-07-01,0.000068,0.000023,0.391694
1,rv,unrestricted_log,894,2022-04-12,2026-07-01,0.000068,0.000023,0.385388
2,rv,ar_log,894,2022-04-12,2026-07-01,0.000072,0.000025,0.612383
3,rv,har_log,894,2022-04-12,2026-07-01,0.000072,0.000026,0.531963
4,rv,gamma_restricted,894,2022-04-12,2026-07-01,0.000208,0.000036,0.423880
5,rv,random_walk,894,2022-04-12,2026-07-01,0.000094,0.000036,0.847949
6,iv,restricted_log,1049,2022-04-12,2026-07-01,0.013461,0.004768,0.143792
7,iv,unrestricted_log,1049,2022-04-12,2026-07-01,0.013418,0.004792,0.146604
8,iv,ar_log,1049,2022-04-12,2026-07-01,0.013461,0.004768,0.143792
9,iv,har_log,1049,2022-04-12,2026-07-01,0.013631,0.004892,0.144788


,target,restricted_vs,loss,DM_stat,p_value,nested_comparison,interpretation
0,rv,unrestricted_log,SE,1.644728,1.003778e-01,True,descriptive for nested pair; standard DM can b...
1,rv,unrestricted_log,AE,0.546022,5.851871e-01,True,descriptive for nested pair; standard DM can b...
2,rv,unrestricted_log,QLIKE,1.523908,1.278857e-01,True,descriptive for nested pair; standard DM can b...
3,rv,ar_log,SE,-3.271555,1.110578e-03,True,descriptive for nested pair; standard DM can b...
4,rv,ar_log,AE,-5.833094,7.602784e-09,True,descriptive for nested pair; standard DM can b...
5,rv,ar_log,QLIKE,-3.031734,2.501745e-03,True,descriptive for nested pair; standard DM can b...
6,rv,har_log,SE,-3.056627,2.305046e-03,False,non-nested predictive comparison
7,rv,har_log,AE,-5.670196,1.925530e-08,False,non-nested predictive comparison
8,rv,har_log,QLIKE,-3.172922,1.560728e-03,False,non-nested predictive comparison
9,rv,random_walk,SE,-2.993564,2.833383e-03,False,non-nested predictive comparison


{'n_test': 1089, 'gaussian_logscore': -1.4471827914352209, 'selected_density_components': 2, 'selected_density_logscore': -1.4414845241684278, 'tail_abs_gt3': 0.0}


## Generated final conclusions and hand-off

The JSON/CSV artifacts, not static prose, are authoritative. Effect sizes are loss ratios relative to the random walk. A model may fail Gaussian or scale diagnostics even when its conditional-mean forecast improves. Marginal p-values—especially nested comparisons—are not promoted to structural conclusions.

In [6]:
summary={"split_date":str(SPLIT.date()),"timing":TIMING,"lag_order":P,"weekday":WEEKDAY,"eq3_timing_assertion":"all regressors earlier than quote-date target","all_forecasts_positive":bool(fc.forecast.gt(0).all()),"results":{}}
for target in ("rv","iv"):
    t=met[met.target.eq(target)].set_index("model"); rw=t.loc["random_walk"]
    summary["results"][target]={m:{"RMSE":float(r.RMSE),"MAE":float(r.MAE),"QLIKE":float(r.QLIKE),"RMSE_vs_RW_pct":float(100*(r.RMSE/rw.RMSE-1)),"QLIKE_vs_RW_pct":float(100*(r.QLIKE/rw.QLIKE-1))} for m,r in t.iterrows()}
summary["diagnostic_failures"]={"return_gaussianity_rejected_training":bool(eq1.JB_p.iloc[0]<ALPHA),"noise_scale_choices":scale_choice,"jump_variant_selected":chosen_components==2}
(OUT/"step2_summary.json").write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))
assert panel.loc[panel.model_day.lt(SPLIT),"model_day"].max()<SPLIT
assert dec["eq3_regressor_rule"]=="all dataframe offsets <= -1"

{
  "split_date": "2022-04-12",
  "timing": "advance",
  "lag_order": 5,
  "weekday": true,
  "eq3_timing_assertion": "all regressors earlier than quote-date target",
  "all_forecasts_positive": true,
  "results": {
    "rv": {
      "restricted_log": {
        "RMSE": 6.789862068247747e-05,
        "MAE": 2.2959560732116035e-05,
        "QLIKE": 0.39169434679798865,
        "RMSE_vs_RW_pct": -27.578449354169653,
        "QLIKE_vs_RW_pct": -53.80682878306015
      },
      "unrestricted_log": {
        "RMSE": 6.775620058106965e-05,
        "MAE": 2.2936473433975358e-05,
        "QLIKE": 0.38538756925019246,
        "RMSE_vs_RW_pct": -27.730356483996708,
        "QLIKE_vs_RW_pct": -54.55059763618263
      },
      "ar_log": {
        "RMSE": 7.198480748191035e-05,
        "MAE": 2.546998485473988e-05,
        "QLIKE": 0.6123828166994276,
        "RMSE_vs_RW_pct": -23.220069444991008,
        "QLIKE_vs_RW_pct": -27.780667417450168
      },
      "har_log": {
        "RMSE": 7.1625267794